# 📈 Superstore 销量预测 (Time Series Forecasting)

## 🎯 商业背景 (Business Context)
> 你是一家美国零售超市的 **Senior Data Analyst**。管理层希望你基于历史订单数据，
> 预测未来一段时间的 **日销售额 (Daily Sales)**，以优化库存管理和运营排班。

## 🧠 数据假设 (Data Assumptions)
- **数据源**: Kaggle Superstore Sales Dataset (`rohitsahoo/sales-forecasting`)
- **时间范围**: 2014-2017 年美国零售订单
- **预测目标**: 日级别 Sales（金额），而非 Quantity
- **挑战**: 数据粒度为交易级别，需聚合为日级别时序

## 🛣️ 路线图 (Roadmap)
1. **🧹 Data Prep**: 加载数据 → 时间解析 → 日聚合 → 缺失日期处理
2. **📊 EDA**: 趋势/周期/分解 → 平稳性检验 (ADF) → ACF/PACF
3. **🔧 Feature Eng**: Lag / Rolling / 日期特征 / 差分 (Differencing)
4. **🔮 Modeling**: Baseline (Naive) → Prophet → XGBoost
5. **📉 Evaluation**: Time Series CV (滚动回测) → MAE/MAPE 对比 → 残差分析
6. **💡 结论与建议**: Actionable Insights for 库存/排班

---

## ⛽️ 函数加油站 (Function Cheat Sheet)

| 函数 | 作用 (大白话) | 常用参数 | SQL 类比 |
| :--- | :--- | :--- | :--- |
| `pd.to_datetime()` | 字符串 → 时间对象 | `format='%d/%m/%Y'` | `CAST(col AS DATE)` |
| `df.set_index().resample()` | 按时间频率聚合 | `'D'`=按天, `'W'`=按周 | `GROUP BY DATE_TRUNC(day, date)` |
| `df.reindex()` | 补齐缺失日期 | `fill_value=0` 或 `.fillna()` | 用日历维表 LEFT JOIN |
| `df['col'].shift(n)` | 滞后 n 期 | `shift(7)` = 上周今天 | `LAG(col, n) OVER()` |
| `df['col'].rolling(n).mean()` | n 期滑动均值 | `min_periods=1` | `AVG() OVER(ROWS n PRECEDING)` |
| `df['col'].diff(n)` | n 阶差分 | `diff(1)` = 今天-昨天 | `col - LAG(col, n)` |
| `seasonal_decompose()` | 趋势+周期+残差分解 | `model='additive'`, `period=7` | 无直接类比 |
| `adfuller()` | ADF 平稳性检验 | 返回 p-value < 0.05 ⇒ 平稳 | 无直接类比 |

> **⚠️ 防泄露铁律**: 构造特征时，`shift()` 的参数必须 ≥ 你的预测步长！
> 例如预测未来 1 天 → `shift(1)` 是最小安全值；预测未来 7 天 → `shift(7)` 才安全。

## 1. 数据导入 (Data Loading)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 4)
plt.style.use("seaborn-v0_8-whitegrid")

# 加载数据
df = pd.read_csv("train.csv")
df.head()

---

## 2. 数据准备 (Data Preparation)
> **任务**: 
> 1. 将 `Order Date` 转为 datetime 类型
> 2. 按日聚合 Sales (Daily Aggregation)
> 3. 处理缺失日期 (Reindex to fill gaps)
> 4. 快速可视化：画出日销售额时间序列折线图

**🤔 思考**: 
- 有些日期可能没有订单（比如节假日），你打算怎么处理？填 0？前向填充？
- 聚合后数据有多少天？时间跨度是多少？

## 3. 探索性分析 (EDA)
> **任务**:
> 1. **时序分解**: 使用 `seasonal_decompose()` 分解趋势 + 周期 + 残差
> 2. **平稳性检验**: 用 ADF 检验判断序列是否平稳
> 3. **ACF/PACF**: 观察自相关和偏自相关图，确定 lag 的选择
> 4. **周度/月度模式**: 不同星期几/不同月份的销售额分布

**🤔 思考**: 
- 这个序列有明显的趋势（涨/跌）吗？有周期性吗（周度？月度？年度？）
- ADF p-value > 0.05 ⇒ 需要差分！你准备如何处理？

## 4. 特征工程 (Feature Engineering)
> **任务**:
> 1. **Lag 特征**: `lag_7`, `lag_14`, `lag_28` 等
> 2. **Rolling 特征**: 7天/14天/28天 滑动均值和标准差
> 3. **日期特征**: dayofweek, month, quarter, is_weekend, is_month_start/end
> 4. **差分特征** (可选): 如果 ADF 不平稳，考虑建模 diff 而非原值
> 5. **Drop NaN**: shift 产生的前几行空值需要清洗

**⚠️ 防泄露提醒**:
- Rolling 特征必须 `shift(1)` 后再 `.rolling()`，否则包含当天数据！
- 如果你预测未来 7 天，lag 特征至少从 `lag_7` 开始

## 5. 建模 (Modeling)

### 5.1 Train/Test Split
> **注意**: 时间序列 **不能随机切分**！必须按时间顺序，前 N% 训练，后 M% 测试。

### 5.2 Baseline: Naive Forecast
> 最简单的基准："明天 = 今天" 或 "下周 = 上周"。
> 如果你的模型连这个都打不过，那就白做了。

### 5.3 Prophet
> Facebook 开源的时序模型，只需要 `ds` (日期) 和 `y` (目标值) 两列。

### 5.4 XGBoost
> 把时序问题变成监督学习回归问题，利用手动构造的特征来预测。
> **进阶**: 尝试 `RandomizedSearchCV` 调参。

## 6. 评估与对比 (Evaluation)
> **任务**:
> 1. 计算每个模型的 **MAE** 和 **MAPE**
> 2. 画 **预测 vs 真实值** 的对比折线图
> 3. **残差分析**: 残差随时间分布 + 残差直方图（是否正态？有无系统性偏差？）
> 4. (进阶) **SHAP**: XGBoost 的特征重要性可视化

**🤔 思考**: 
- XGBoost 在哪些区间表现更好？哪些更差？
- 如果残差有系统性偏差 → 说明什么？如何改进？

## 7. 结论与建议 (Conclusion & Recommendations)
> **任务**: 写出 3-5 条 **Actionable Insights**
> - 哪个模型表现最好？MAE 是多少？意味着什么？
> - 对库存管理有什么建议？（如安全库存天数）
> - 对运营排班有什么建议？（如周末是否需要加人？）
> - 如果有更多数据（促销、天气），模型还能怎么改进？